# Compare offline and online-binned mixed layer tracer budgets

This notebook contains code to compare the online method (most accurate) with offline methods based on monthly and daily binning of heat budget diagnostics into the mixed layer.

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import string
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=7)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures/Online_Offline_Comparison')

# Load data

### Define paths, region to analyse and time period to analyse

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'          # Location of simulation output
tmp_folder = base + 'post_processed_diags/'                                                  # Temp folder where post-processed diagnostics (pre-processed budgets) are contained

# Choose year to consider:
output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors
base2 = base + 'output%03d/ocean/' % output

# Climatology:
clim_str = 'output336-365' # 336-365 = 1989-2018
clim_label = '1989-2018'

# Region to load:
reg = [-270, -70, -70, 60] # Pacific

# Time period to load:
#times = slice('2019-01-01','2019-01-31')#lice(None,None)
#times_snap = slice('2019-01-01','2019-02-01') # Note; this must be 1 more than times.
times = slice(None,None)
times_snap = slice(None,None) # Note; this must be 1 more than times.

# Chunks to use:
chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

# Set constants:
rho0 = 1035.
Cp = 3992.10322329649

In [ ]:
# Grid file for area-averaging:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

### Load pre-computed MLT budgets

In [ ]:
# Online monthly averages:
mlt_budget_online = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_output%03d_monthly_mean.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times)
mlt_budget_online_clim = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_online_' + clim_str + '_monthly_mean.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).assign_coords({'time':mlt_budget_online.time})

# Monthly offline:
mlt_budget_monthly_offline = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_monthly_offline_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times).drop_vars(['st_ocean'])
mlt_budget_monthly_offline_clim = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_monthly_offline_' + clim_str + '.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).drop_vars(['st_ocean']).assign_coords({'time':mlt_budget_monthly_offline.time})

# Daily offline:
mlt_budget_daily_offline = xr.open_dataset(tmp_folder + 'mlt_budget_stavg_daily_offline_output%03d_monthly_mean.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times).drop_vars(['st_ocean'])

In [ ]:
# Add some extra variables for convenience:
for budget in [mlt_budget_online,mlt_budget_online_clim,mlt_budget_monthly_offline,mlt_budget_monthly_offline_clim,mlt_budget_daily_offline]:
    budget['surface_flux_absorbed'] = budget['surface_flux'] + budget['sw_pen']
    budget['vert_all'] = budget['surface_flux_absorbed'] + budget['vert_mixing']

for budget in [mlt_budget_monthly_offline,mlt_budget_monthly_offline_clim,mlt_budget_daily_offline]:
    budget['mlt_tendency'] = np.nan*budget['advection']
    budget['entrainment'] = np.nan*budget['advection']

# Plots comparing offline and daily budgets

#### Spatial plots comparing monthly, daily offline and daily:

In [ ]:
regs = {'Warm Pool':[-200, -160, -10, 10],
        'Tasman Sea':[-205, -190, -40, -25],
        'Cold Tongue':[-150,-90,-5,5],
       }

In [ ]:
# Region selection:
fig = plt.figure(figsize=(30,15))
mlt_budget_monthly_offline['vert_mixing'].mean('time').plot()
for key in regs.keys():
    sreg = regs[key]
    plt.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
plt.gca().set_facecolor('k')

In [ ]:
# Spatial plots:

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux_absorbed','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','Shortwave penetration']

# # Raw budget terms:
# fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(15,7),layout='constrained')
# budgets = [mlt_budget_online.sel(time='2019-11'),#mean('time'),
#            mlt_budget_monthly_offline.sel(time='2019-11'),#mean('time'),
#            mlt_budget_daily_offline.sel(time='2019-11'),#mean('time'),
# #           mlt_budget_online.mean('time') - mlt_budget_monthly_offline.mean('time'),
# #           mlt_budget_online.mean('time') - mlt_budget_daily_offline.mean('time')
#           ]
# budget_labels = ['Online','Monthly Offline','Daily Offline']
# unit_conv = 86400*30.5
# clims = [5,2,2,5,5,10]
# fname = 'MLT_budget_Nov2019_Online_Offline_Comparison_Pacific.png'

# Anomaly budget terms:
fig, axs = plt.subplots(nrows=3, ncols=6, figsize=(15,7),layout='constrained')
budgets = [mlt_budget_online.sel(time='2019-11').mean('time') - mlt_budget_online_clim.sel(time='2019-11').mean('time'),
           mlt_budget_monthly_offline.sel(time='2019-11').mean('time') - mlt_budget_monthly_offline_clim.sel(time='2019-11').mean('time'),
           mlt_budget_online.sel(time='2019-11').mean('time') - mlt_budget_online_clim.sel(time='2019-11').mean('time') - mlt_budget_monthly_offline.sel(time='2019-11').mean('time') + mlt_budget_monthly_offline_clim.sel(time='2019-11').mean('time')
          ]
budget_labels = ['Online','Monthly Offline','Online - Monthly Offline']
unit_conv = 86400*30.5 
clims = [2,1,1,2,4,4]
fname = 'MLT_budget_Nov2019_Online_Offline_Comparison_Pacific_Anomalies.png'

for i in range(len(budgets)):
    for j, var in enumerate(vars):
        if i == 0:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',cbar_kwargs={'label':labels[j] + ' ($^\circ$C/month)','shrink':0.7,'location':'top','ticks':[-clims[j],0,clims[j]]})
        else:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',add_colorbar=False)
        axs[i][j].set_ylabel('')
    axs[i][0].set_ylabel(budget_labels[i])
    
for key in regs.keys():
    sreg = regs[key]
    for ax in axs.reshape(-1)[2:6]:
        ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k',linewidth=0.5)

j = 0
for i, ax in enumerate(axs.reshape(-1)):
    ax.set_ylim([-70,60])
    ax.set_xlabel('')
    ax.set_title('')
    ax.set_facecolor('k')
    ax.set_yticks([-60,-30,0,30,60])
    ax.set_xticks([-250,-180,-110])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    if i not in [6,7,12,13]:
        ax.text(-269,59,f'({string.ascii_lowercase[j]})',verticalalignment='top',color='w')
        j += 1

# Turn off the tendency and entrainment axes for the offline budgets:
axs[1][0].set_axis_off()
axs[1][1].set_axis_off()
axs[2][0].set_axis_off()
axs[2][1].set_axis_off()
axs[1][2].set_ylabel(budget_labels[1])
axs[2][2].set_ylabel(budget_labels[2])
axs[0][0].set_yticklabels(['60S','30S','0','30N','60N'])
for ax in [axs[0][0],axs[0][1],axs[2][2],axs[2][3],axs[2][4],axs[2][5]]:
    ax.set_xticklabels(['250W','180W','110W'])

plt.savefig(fname,dpi=300)

In [ ]:
# Compute spatial averages:
area = ds_grid.area_t.load()
budgets = {}
budgets_anom = {}

months = {
    'Warm Pool': '2019-11',
    'Tasman Sea': '2019-11',
    'Cold Tongue': '2019-11',
}

def area_average(budget,reg):

    total_area = area.sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])
    budget_av = (budget*area).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).mean(['xt_ocean','yt_ocean'])/total_area
    return(budget_av)

for key in tqdm(regs.keys()):
    sreg = regs[key]
    budgets[key] = [area_average(mlt_budget_online,sreg).sel(time=months[key],drop=True).drop_vars(['time']).load(),
                   area_average(mlt_budget_monthly_offline,sreg).sel(time=months[key],drop=True).drop_vars(['time']).load(),
               area_average(mlt_budget_daily_offline,sreg).sel(time=months[key],drop=True).drop_vars(['time']).load()
              ]
    budgets_anom[key] = [area_average(mlt_budget_online-mlt_budget_online_clim,sreg).sel(time=months[key],drop=True).drop_vars(['time']).load(),
                         area_average(mlt_budget_monthly_offline-mlt_budget_monthly_offline_clim,sreg).sel(time=months[key],drop=True).drop_vars(['time']).load()               
              ]

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(22, 15))


unit_conv = 86400 * 30.5
bar_width = 0.3

# ---- term groupings ----
vars_grouped = [
    'fixedh_tendency',
    'advection',
    'vert_all'
]
labs_grouped = [
    'Total',
    'Advection',
    'Surface fluxes\n+ Vertical Mixing'
]

vars_split = [
    'vert_mixing',
    'surface_flux_absorbed',
    'sw_pen'
]
labs_split = [
    'Vertical Mixing',
    'Surface fluxes',
    'SW penetration'
]

def plot_budget(ax, ds, classes, variables, labels):
    df = ds.to_dataframe().reset_index()
    x = np.arange(len(variables))

    for i, cls in enumerate(classes):
        values = [
            df[df['class'] == cls][var].values[0] * unit_conv
            for var in variables
        ]
        ax.bar(x + i * bar_width, values, width=bar_width, label=cls)

    ax.set_xticks(x + bar_width / 2)
    ax.set_xticklabels(labels)
    ax.grid(axis='y')

# ---- main plotting loop ----
for r, key in enumerate(budgets.keys()):

    # ===== RAW =====
    classes_raw = ['Online', 'Monthly Offline', 'Daily Offline']
    ds_raw = (
        xr.concat(budgets[key], dim='class')
        .assign_coords({'class': classes_raw})
    )

    plot_budget(axes[r, 0], ds_raw, classes_raw, vars_grouped, labs_grouped)
    plot_budget(axes[r, 1], ds_raw, classes_raw, vars_split, labs_split)

    # ===== ANOMALIES =====
    classes_anom = ['Online', 'Monthly Offline']
    ds_anom = (
        xr.concat(budgets_anom[key], dim='class')
        .assign_coords({'class': classes_anom})
    )

    plot_budget(axes[r, 2], ds_anom, classes_anom, vars_grouped, labs_grouped)
    plot_budget(axes[r, 3], ds_anom, classes_anom, vars_split, labs_split)

    # ---- row labels / titles ----
    axes[r, 0].set_ylabel("$^\circ$C/month")
    axes[r, 0].set_title(
        f"{key}\nRaw",
        loc='left'
    )
    axes[r, 2].set_title(
        f"{key}\nAnomalies",
        loc='left'
    )

# ---- column titles ----
#axes[0, 0].set_title("Raw – grouped terms")
#axes[0, 1].set_title("Raw – split terms")
#axes[0, 2].set_title("Anomalies – grouped terms")
#axes[0, 3].set_title("Anomalies – split terms")
for ax in axes.flat:
    ax.axhline(0, color='k', linewidth=1.5, alpha=0.5)
# Column 0: Raw – grouped terms
for ax in axes[:, 0]:
    ax.set_ylim([-0.25, 2.5])

# Column 1: Raw – split terms
for ax in axes[:, 1]:
    ax.set_ylim([-7, 7])

# Column 2: Anomalies – grouped terms
for ax in axes[:, 2]:
    ax.set_ylim([-0.25, 0.75])

# Column 3: Anomalies – split terms
for ax in axes[:, 3]:
    ax.set_ylim([-1, 1.5])

# ---- legend (single, shared) ----
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, title="Budget", loc="upper center", ncol=3)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(
    "MLT_budget_Nov2019_Online_Offline_Comparison_Bar.png",
    dpi=250
)


In [ ]:
# Bar plots (for a single set of budgets):
fig, axes = plt.subplots(nrows=3,ncols=1,figsize=(13, 15))

months = {'Warm Pool':range(12),
        'Tasman Sea':[9,10,11,0,1,2],
        'Cold Tongue':range(12),
       }
month_lab = {'Warm Pool':'Annual',
        'Tasman Sea':'Oct-Dec, Jan-Feb',
        'Cold Tongue':'Annual'}

# Bar plots across specific regions:
for i, key in enumerate(budgets.keys()):
    ax = axes[i]

    # Raw:
    #classes = ['Monthly Offline','Daily Offline','Online']
    #ds = xr.concat(budgets[key],dim='class').assign_coords({'class':classes}).isel(time=months[key]).mean('time')
    #labelra = ' '

    # Anomalies:
    classes = ['Monthly Offline','Online']
    ds = xr.concat(budgets_anom[key],dim='class').assign_coords({'class':classes}).isel(time=months[key]).mean('time')
    labelra = ' Anomalies '
    
    variables = ['fixedh_tendency','advection','vert_mixing','surface_flux_absorbed','sw_pen','vert_all']
    labels = ['Total','Advection','Vertical Mixing','Surface fluxes','SW penetration','Surface fluxes+ \nSW penetration \n+ Vertical Mixing']
    
    # Convert to DataFrame for easier plotting
    df = ds.to_dataframe()
    
    # Reset index to get 'class' as a column
    df = df.reset_index()
    
    # Parameters
    num_vars = len(variables)
    num_classes = len(classes)
    bar_width = 0.25
    x = np.arange(num_vars)  # One x position per variable
    
    unit_conv = 86400*30.5
    # Create figure
    
    # Plot each class as a separate bar group
    for i, cls in enumerate(classes):
        # Get values for this class across all variables
        values = [df[df['class'] == cls][var].values[0]*unit_conv for var in variables]
        
        # Offset x positions for each class
        ax.bar(x + i * bar_width, values, width=bar_width, label=cls)
    
    # Formatting
    ax.set_xticks(x + bar_width)
    ax.set_xticklabels(labels)
    ax.set_ylabel("$^\circ$C/month")
    ax.set_title(key + ' Mixed Layer Temperature Budget' + labelra +month_lab[key] + ' 2019')
    ax.legend(title="Budget")
    ax.grid()
    plt.tight_layout()
plt.savefig('MLT_budget_2019_Online_Offline_Comparison_Bar_Anomalies.png',dpi=250)

In [ ]:
# Time series across specific regions:
fig, axs = plt.subplots(nrows=len(regs.keys()), ncols=1, figsize=(12,12))

vars = ['advection','vert_mixing','surface_flux_absorbed','sw_pen']
labels = ['Advection','Vertical Mixing','Surface fluxes','Shortwave penetration']
cols = ['k','r','b','g','m','c']
typs= ['-','--',':']
budget_labels = ['Monthly Offline','Daily Offline','Online']
unit_conv = 86400*30.5

for k, key in enumerate(regs.keys()):
    sreg = regs[key]
    for i in range(len(budgets)):
        for j, var in enumerate(vars):
            if j == 0 and k == 0:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=budget_labels[i])
            elif i == 0 and k == 1:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2,label=labels[j])
            else:
                (budgets[key][i][var]*unit_conv).plot(ax=axs[k],color=cols[j],linestyle=typs[i],linewidth=2)
    axs[k].set_title(key)
axs[0].legend()
axs[1].legend()
plt.tight_layout()
#plt.savefig('MLT_budget_2023_Online_Offline_Comparison_Time_Series.png',dpi=300)

# Compute offline budgets

The following diagnostics are needed to do the offline binning:
- Closed 3D heat budget
- `dzt` and `mld` time-averages to bin the 3D diagnostics into the mixed layer.
Various diagnostics for the correction terms:
- `temp_in_mld` time average
- `temp` time average to compute temperature at base of mixed layer, or alternatively `temp_at_mlb` (although this is an online diagnostic so using an interpolation of `temp` offline is probably a bit more representative of what an offline calculation would be).
- `eta_t`, `pme_river`, `eta_t_tendency`, `eta_smoother` time averages
- `ht` grid depths

Below, there is code to compute both monthly and daily resolution offline budgets. However, for computation across a production run there are also two sets of scripts:
1. For the monthly offline calculations across a large number of years, please see the PBS processing scripts `process_offline_monthly_budget_year.sub` and `spawn_process_offline_monthly_budget.py`.
2. FOr the daily offline calculations, instead used PBS processing scripts `process_offline_daily_budget_month.sub` and `spawn_process_offline_daily_budget.py`

Some extra command line processing steps for the daily offline calculation:

Make record dimension:
`for f in mlt_budget_stavg_daily_offline_output366_month*.nc; do echo $f; ncks -O --mk_rec_dmn time $f $f; done`

Concatenate files:
`ncrcat mlt_budget_stavg_daily_offline_output366_month*.nc mlt_budget_stavg_daily_offline_output366.nc`

## First, load data

In [ ]:
# Grid file (for ht):
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)

In [ ]:
# Standard daily diagnostics:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

# 3D budget data:
ds_day_budget_3d = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Daily snapshots:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget_3d = ds_day_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_3d.time.values]})
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget_3d.average_DT.data = ds_day_budget_3d.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget_3d = ds_day_budget_3d.sel(time=times)
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

In [ ]:
# Standard monthly diagnostics:
ds_mon = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = ds_mon.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon.time.values]})
ds_mon.average_DT.data = ds_mon.average_DT*np.timedelta64(1,'D')
ds_mon = ds_mon.sel(time=times)

# 3D budget data:
ds_mon_budget_3d = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc',decode_times=False,chunks=chunks3D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Monthly snapshots:
ds_mon_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_mon_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon_snapshot = xr.concat([ds_mon_snapshot_m1.isel(time=-1),ds_mon_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_mon_budget_3d = ds_mon_budget_3d.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_budget_3d.time.values]})
ds_mon_snapshot = ds_mon_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_mon_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_mon_budget_3d.average_DT.data = ds_mon_budget_3d.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_mon_budget_3d = ds_mon_budget_3d.sel(time=times)
ds_mon_snapshot = ds_mon_snapshot.sel(time=times_snap)

## Define functions to compute offline mixed layer temperature budget

In [ ]:
def mixed_layer_bin_offline(ds_budget,ds_budget_2d,mld,dzt):
    """
    Bin raw ("Eulerian") 3D (and 2D-surface layer only) time-averaged budget terms into the time-averaged mixed layer

    Inputs:
    ds_budget -> dataset containing time-averaged 3D budget quantities
    ds_budget_2d -> dataset containing time-averaged 2D (surface layer only) budget quantities
    mld -> dataarray containing time-averaged mixed layer depth
    dzt -> datarray containing time-averaged grid cell thicknesses

    Outputs:
    ds_budget_in_mld -> dataset containing budget terms binned into the mixed layer, in units of degC/second
    """

    # Sum over mixed layer:
    ds_budget_in_mld = ds_budget.isel(st_ocean=0,drop=True).copy(deep=True)
    for var in list(ds_budget_2d.data_vars):
        ds_budget_in_mld[var] = xr.zeros_like(ds_budget['fixedh_tendency']).isel(st_ocean=0,drop=True).copy(deep=True)
    for ti in range(len(ds_budget.time)): # Loop over time
        dzt_ti = dzt.isel(time=ti).load()       
        dzt_ti_bot = dzt_ti.cumsum('st_ocean')  # Depth (from free surface) of bottom of each cell
        ds_budget_ti = ds_budget.isel(time=ti).load()
        mld_ti = mld.isel(time=ti).load()
        for var in list(ds_budget.data_vars):
            ds_budget_in_mld[var][ti,:,:] = ds_budget_ti[var].where(dzt_ti_bot<mld_ti).sum('st_ocean')   # Include all of cells that lie completely in the mixed layer
            for k in range(len(ds_budget_ti.st_ocean)-1):
                ds_budget_in_mld[var][ti,:,:] += xr.where(np.logical_and(dzt_ti_bot[k,:,:]>mld_ti,dzt_ti_bot[k+1,:,:]<mld_ti),((mld_ti-dzt_ti_bot[k,:,:])/dzt_ti[k,:,:])*ds_budget_ti[var][k,:,:],0.) # Include only a fraction of cells that lie partially within the mixed layer
            ds_budget_in_mld[var][ti,:,:] = ds_budget_in_mld[var][ti,:,:]/rho0/Cp/mld_ti # Convert units from Wm-2 to degC/sec

        # Add 2D surface layer variables:
        surf_frac = xr.where(dzt_ti_bot[0,:,:]>mld_ti,mld_ti/dzt_ti[0,:,:],1.)            # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, take only that fraction from the 2D variables
        for var in list(ds_budget_2d.data_vars):
            ds_budget_in_mld[var][ti,:,:] = (surf_frac*ds_budget_2d[var][ti,:,:]/rho0/Cp/mld_ti).load()

    # Compute residual for check:
    ds_budget_in_mld['residual'] = ds_budget_in_mld['fixedh_tendency'] - ds_budget_in_mld[list(ds_budget_in_mld.data_vars)[1:]].to_array().sum('variable')
        
    return(ds_budget_in_mld)

def compute_tendency_entrainment(mlt_budget,mlt_snap):
    """
    Compute tendency term from snapshots and entrainment by residual.
    """

    mlt_snap = mlt_snap.transpose(*mlt_budget['fixedh_tendency'].dims)

    # Compute mlt tendency by taking time derivative of snapshot mlt:
    mlt_budget['mlt_tendency'] = xr.zeros_like(mlt_budget.fixedh_tendency).copy(deep=True)
    mlt_budget['mlt_tendency'].data = mlt_snap.isel(time=slice(1,None)).values - mlt_snap.isel(time=slice(0,-1)).values
    # mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Note: Depending on time decoding this line may change
    DT = xr.zeros_like(mlt_budget['mlt_tendency'].time)
    DT.data = (mlt_snap.time.isel(time=slice(1,None)).values-mlt_snap.time.isel(time=slice(0,-1)).values)/np.timedelta64(1,'s')
    mlt_budget['mlt_tendency'] = mlt_budget['mlt_tendency']/DT

    # Compute entrainment by residual:
    mlt_budget['entrainment'] = -(mlt_budget['fixedh_tendency'] - mlt_budget['mlt_tendency'])

    return(mlt_budget)

def compute_corrections(mlt_budget,ds_std,D,save_cor=False,use_temp_at_mlb=False):
    """
    Compute corrections to advection, surface mass flux and total tendency terms in offline budgets.

    Inputs (all time averages):
    - mlt_budget               - grouped budget in units degC/sec
    - ds_std['pme_river']      - surface mass flux (kg m-2 s-1)
    - ds_std['temp_in_mld']    - mixed layer temperature (kg m-3 deg C)
    - ds_std['eta_t_tendency'] - deta/dt (kg m-2 s-1)
    - ds_std['eta_t']          - free surface height (m)
    - D                        - ocean depth (m)
    - ds_std['mld']            - mixed layer depth 
    - ds_std['dzt']            - layer thicknesses (for computing fraction of surface layer)
    - ds_std['temp']           - 3D temperature (for computing temperature at base of mixed layer)
    - save_cor                 - whether to save the correction terms individually for testing
    
    """

    # Extract variables and do some unit conversions:
    mlt = ds_std.temp_in_mld.load()/rho0  # DegC
    pme = ds_std.pme_river.load()/rho0    # ms-1
    eta = ds_std.eta_t.load()             # m
    detadt = ds_std.eta_t_tendency.load() # ms-1
    eta_smooth = ds_std.eta_smoother.load() # ms-1
    H = ds_std.mld.load()                         # m
    dzs = ds_std.dzt.isel(st_ocean=0).load()      # m
    if use_temp_at_mlb:      # Use online computed temp_at_mlb
        tmlb = ds_std.temp_at_mlb.load()
    else:      # Compute temp_at_mlb from time-averaged 3D temperature and mld
        t3d = ds_std.temp.load()              # 3D temperature for tmlb
        if t3d.max() > 100.: t3d -= 273.15    # Convert to celsius if needed
    
        # Compute entrained tracer for adv_cor2 (note, since H is a thickness of water, we can interpolate using st_ocean, not the time and space dependent z):
        tmlb = compute_tmlb(t3d,H)
    
    # In regions where the mixed layer depth is shallower than the thickness of the surface grid cell, 
    # take only that fraction from the 2D variables (which all these corrections are):
    surf_frac = xr.where(dzs>H,H/dzs,1.)
    
    # Compute corrections: 
    pme_cor = -pme*mlt/H*surf_frac                             # ms-1 degC / m = degC s-1
    eta_smoother_cor = -eta_smooth*mlt/H*surf_frac
    adv_cor1 = (-detadt + pme + eta_smooth)*mlt/H*surf_frac
    adv_cor2 = detadt*tmlb*(1-H/(D+eta))/H*surf_frac
    if save_cor:
        mlt_budget['pme_cor'] = pme_cor
        mlt_budget['eta_smoother_cor'] = eta_smoother_cor
        mlt_budget['adv_cor1'] = adv_cor1
        mlt_budget['adv_cor2'] = adv_cor2
        
    # Apply corrections:
    mlt_budget['advection'] += adv_cor1 + adv_cor2
    mlt_budget['surface_flux'] += pme_cor + eta_smoother_cor
    mlt_budget['fixedh_tendency'] += adv_cor1 + adv_cor2 + pme_cor + eta_smoother_cor
    
    return(mlt_budget)

def compute_tmlb(t3d,H):
    """
    Linearly interpolate t3d to H (I don't understand why xarray's interp finds this so hard!)
    """
    
    tmlb = np.nan*xr.zeros_like(t3d.isel(st_ocean=0)).values
    
    tmlb = xr.where(H<t3d.st_ocean[0],t3d.isel(st_ocean=0).values,np.nan)
    tmlb = xr.where(H>t3d.st_ocean[-1],t3d.isel(st_ocean=-1).values,tmlb)
    
    for s in range(len(t3d.st_ocean)-1):
        z = t3d.st_ocean[s]
        zp1 = t3d.st_ocean[s+1]
        between_pts = np.logical_and(z < H,zp1 > H)
        tz = t3d.isel(st_ocean=s)
        tzp1 = t3d.isel(st_ocean=s+1)
        tmlb = xr.where(between_pts,tzp1*(H - z)/(zp1 - z) + tz*(zp1 - H)/(zp1 - z),tmlb)
    return(tmlb)

def compute_offline_budget_ti(ds_budget_3d,ds_std,ht,use_temp_at_mlb=False):
    """
    Compute offline budget for a single time slice
    """

    # Group terms and load data (we do this before doing the binning to save unneccessary compute):
    ds_budget_3d_reduced = ds_budget_3d['temp_tendency'].load().rename('fixedh_tendency').to_dataset()
    for var in bud_var_grps.keys():
        ds_budget_3d_reduced[var] = ds_budget_3d[bud_var_grps[var][0]].load()
        if (len(bud_var_grps[var])>1):
            for raw_var in bud_var_grps[var][1:]:
                ds_budget_3d_reduced[var] += ds_budget_3d[raw_var].load()
    
    ds_budget_3d_reduced_2d = ds_budget_3d[bud_2d_vars].load()
    
    # Do binning into mixed layer:
    bud = mixed_layer_bin_offline(ds_budget_3d_reduced,ds_budget_3d_reduced_2d,ds_std.mld.load(),ds_std.dzt.load())
    # Add 2D vars to sbc term:
    for var in bud_2d_vars:
        bud['surface_flux'] += bud[var]
        bud = bud.drop_vars([var])
    
    # Compute offline correction terms and apply them:
    bud = compute_corrections(bud,ds_std,ht,save_cor=False,use_temp_at_mlb=use_temp_at_mlb)

    # Compute tendency and entrainment (not really correct for offline, so we don't do it):
    #bud = compute_tendency_entrainment(bud,ds_snapshot.temp_in_mld/rho0)
    
    return(bud)    

## Compute daily and monthly offline binned mixed layer temperature budgets (standard averaging):

In [ ]:
# Offline budget terms grouping:
bud_var_grps = {'advection':['temp_advection','temp_submeso','temp_vdiffuse_k33','neutral_diffusion_temp','neutral_gm_temp'],
                'vert_mixing':['temp_vdiffuse_diff_cbt','temp_nonlocal_KPP'],
                'surface_flux':['temp_vdiffuse_sbc','frazil_3d','temp_rivermix'],
                'sw_pen':['sw_heat']}
bud_2d_vars = ['temp_eta_smooth','sfc_hflux_pme']

In [ ]:
# Compute monthly offline budget, by month:
# NOTE: For production please see the PBS processing scripts `process_offline_monthly_budget_year.sub` 
# and `spawn_process_offline_monthly_budget.py`.

# Add sfc_hflux_pme into budget ds from standard monthly diagnostics file:
ds_mon_budget_3d['sfc_hflux_pme'] = ds_mon['sfc_hflux_pme']

# ht for zstar calc:
ht = ds_grid.ht.load()

mlt_budget_stavg_monthly_offline_uncat = []

# Loop over month:
for ti in tqdm(range(len(ds_mon_budget_3d.time))):

    bud = compute_offline_budget_ti(ds_mon_budget_3d.isel(time=slice(ti,ti+1)),
                                    ds_mon.isel(time=slice(ti,ti+1)),
                                    ht)
    
    mlt_budget_stavg_monthly_offline_uncat.append(bud)

mlt_budget_stavg_monthly_offline = xr.concat(mlt_budget_stavg_monthly_offline_uncat,dim='time')

In [ ]:
# Save to file:
mlt_budget_stavg_monthly_offline.to_netcdf(tmp_folder + 'mlt_budget_stavg_monthly_offline_2019.nc')

In [ ]:
# Compute daily offline budget, by day:

# ht for zstar calc:
ht = ds_grid.ht.load()

# Add some vars to standard outputs from budget output:
ds_day['dzt'] = ds_day_budget_3d.dzt
ds_day['eta_t_tendency'] = ds_day_budget_3d.eta_t_tendency
ds_day['eta_smoother'] = ds_day_budget_3d.eta_smoother

mlt_budget_stavg_daily_offline_uncat = []

# Loop over day:
for ti in tqdm(range(len(ds_day_budget_3d.time))):

    bud = compute_offline_budget_ti(ds_day_budget_3d.isel(time=slice(ti,ti+1)),
                                    ds_day.isel(time=slice(ti,ti+1)),
                                    ht,use_temp_at_mlb=True)
    
    mlt_budget_stavg_daily_offline_uncat.append(bud)

mlt_budget_stavg_daily_offline = xr.concat(mlt_budget_stavg_daily_offline_uncat,dim='time')

In [ ]:
# Save to file:
fname = tmp_folder + 'mlt_budget_stavg_daily_offline_output%03d.nc' % output
mlt_budget_stavg_daily_offline.to_netcdf(fname)